In [1]:
from pathlib import Path
import pandas as pd

MERGED_DIR = Path("outputs/final_50000/merged")

for file in MERGED_DIR.glob("*.csv"):
    print(file.name)

reviews_50000_findings.csv
reviews_50000_review_summary.csv


## loading and checking the two files

In [2]:
findings_file = MERGED_DIR / "reviews_50000_findings.csv"
reviews_file = MERGED_DIR / "reviews_50000_review_summary.csv"

findings_df = pd.read_csv(findings_file)
reviews_df = pd.read_csv(reviews_file)

print("Findings table:", findings_df.shape)
print("Reviews table:", reviews_df.shape)

print("\nFindings columns:")
print(findings_df.columns.tolist())

print("\nReviews columns:")
print(reviews_df.columns.tolist())

Findings table: (201623, 19)
Reviews table: (50000, 13)

Findings columns:
['source_row', 'part_number', 'listing_id', 'comment_id', 'date', 'year', 'listing_comment_count', 'listing_activity', 'finding_number', 'aspect', 'object', 'observation', 'aspect_score', 'severity_score', 'evidence_quote', 'evidence_quote_exact', 'finding_category', 'manual_review', 'comments_clean']

Reviews columns:
['source_row', 'part_number', 'listing_id', 'comment_id', 'date', 'year', 'listing_comment_count', 'listing_activity', 'comments_original', 'comments_clean', 'finding_count', 'batch_id', 'attempt']


## data-quality check for the extraction problems

In [3]:
print("===== REVIEW-LEVEL CHECK =====")

print("Total reviews:", len(reviews_df))
print("Unique comment IDs:", reviews_df["comment_id"].nunique())
print("Duplicate comment IDs:", reviews_df["comment_id"].duplicated().sum())

print("\nReviews by part:")
print(reviews_df["part_number"].value_counts().sort_index())

print("\n===== FINDING-LEVEL CHECK =====")

print("Total findings:", len(findings_df))
print("Sum of finding_count from review table:", reviews_df["finding_count"].sum())

print("\nEvidence quote exact match:")
print(findings_df["evidence_quote_exact"].value_counts(dropna=False))

print("\nManual review:")
print(findings_df["manual_review"].value_counts(dropna=False))

print("\n===== SCORE / SCHEMA CHECK =====")

print(
    "Aspect scores outside -5 to 5:",
    (~findings_df["aspect_score"].between(-5, 5)).sum()
)

print(
    "Negative findings missing severity:",
    (
        (findings_df["aspect_score"] < 0)
        & (findings_df["severity_score"].isna())
    ).sum()
)

print(
    "Non-negative findings with severity:",
    (
        (findings_df["aspect_score"] >= 0)
        & (findings_df["severity_score"].notna())
    ).sum()
)

===== REVIEW-LEVEL CHECK =====
Total reviews: 50000
Unique comment IDs: 50000
Duplicate comment IDs: 0

Reviews by part:
part_number
1    10000
2    10000
3    10000
4    10000
5    10000
Name: count, dtype: int64

===== FINDING-LEVEL CHECK =====
Total findings: 201623
Sum of finding_count from review table: 201623

Evidence quote exact match:
evidence_quote_exact
True     196837
False      4786
Name: count, dtype: int64

Manual review:
manual_review
False    191896
True       9727
Name: count, dtype: int64

===== SCORE / SCHEMA CHECK =====
Aspect scores outside -5 to 5: 0
Negative findings missing severity: 0
Non-negative findings with severity: 0


## check the warning flags

In [4]:
print("Manual-review findings:", findings_df["manual_review"].sum())

print(
    "Manual-review rows that are NOT Other:",
    (
        findings_df["manual_review"]
        & (findings_df["aspect"] != "Other")
    ).sum()
)

print(
    "Other rows not marked for manual review:",
    (
        (findings_df["aspect"] == "Other")
        & (~findings_df["manual_review"])
    ).sum()
)

print("\nQuote warnings by aspect:")
print(
    findings_df.loc[
        ~findings_df["evidence_quote_exact"],
        "aspect"
    ].value_counts()
)

Manual-review findings: 9727
Manual-review rows that are NOT Other: 0
Other rows not marked for manual review: 0

Quote warnings by aspect:
aspect
Overall stay             2696
Location                  428
Communication             283
Amenities                 250
Aesthetics and design     207
Comfort                   189
Cleanliness               153
Space and capacity        106
Quietness                  94
Property condition         71
Other                      70
Check-in                   62
Safety and security        50
Views                      30
Accuracy of listing        29
Privacy                    29
Value for money            22
Check-out                  17
Name: count, dtype: int64


# First actual business-analysis question

In [5]:
aspect_summary = (
    findings_df[findings_df["aspect"] != "Other"]
    .groupby("aspect")
    .agg(
        finding_count=("aspect", "size"),
        average_score=("aspect_score", "mean")
    )
    .sort_values("finding_count", ascending=False)
)

aspect_summary

,finding_count,average_score
aspect,,
Overall stay,38065,3.979798
Amenities,23542,2.802311
Location,23213,3.438763
Communication,19549,3.875032
Cleanliness,15249,3.106171
Aesthetics and design,12808,3.661540
Comfort,12633,3.238423
Quietness,9467,2.706137
Space and capacity,9400,3.051277


In [6]:
findings_df["finding_category"].value_counts(dropna=False)

finding_category
Strength    185680
Problem      15181
Neutral        762
Name: count, dtype: int64

## Strengths and Problems distribution across each aspect

In [7]:
aspect_category = (
    findings_df[findings_df["aspect"] != "Other"]
    .groupby(["aspect", "finding_category"])
    .size()
    .unstack(fill_value=0)
)

aspect_category

finding_category,Neutral,Problem,Strength
aspect,,,
Accuracy of listing,8,1306,4357
Aesthetics and design,18,154,12636
Amenities,93,3010,20439
Check-in,36,568,4088
Check-out,5,85,955
Cleanliness,6,1590,13653
Comfort,16,863,11754
Communication,16,629,18904
Location,151,770,22292


## problem rate for every aspect

In [8]:
aspect_category["total"] = aspect_category.sum(axis=1)

aspect_category["problem_rate"] = (
    aspect_category["Problem"] / aspect_category["total"]
)

aspect_category["strength_rate"] = (
    aspect_category["Strength"] / aspect_category["total"]
)

aspect_category.sort_values("problem_rate", ascending=False)

finding_category,Neutral,Problem,Strength,total,problem_rate,strength_rate
aspect,,,,,,
Property condition,35,2147,1420,3602,0.596058,0.394225
Accuracy of listing,8,1306,4357,5671,0.230294,0.768295
Value for money,19,358,2011,2388,0.149916,0.842127
Safety and security,10,613,3601,4224,0.145123,0.852509
Quietness,29,1292,8146,9467,0.136474,0.860463
Amenities,93,3010,20439,23542,0.127857,0.868193
Check-in,36,568,4088,4692,0.121057,0.871270
Privacy,42,355,2550,2947,0.120461,0.865287
Cleanliness,6,1590,13653,15249,0.104269,0.895337


## average severity of the problems

In [9]:
problem_severity = (
    findings_df[
        (findings_df["aspect"] != "Other")
        & (findings_df["finding_category"] == "Problem")
    ]
    .groupby("aspect")["severity_score"]
    .mean()
)

priority_summary = aspect_category.copy()

priority_summary["average_problem_severity"] = problem_severity

priority_summary = priority_summary.sort_values(
    "Problem",
    ascending=False
)

priority_summary

finding_category,Neutral,Problem,Strength,total,problem_rate,strength_rate,average_problem_severity
aspect,,,,,,,
Amenities,93,3010,20439,23542,0.127857,0.868193,1.952824
Property condition,35,2147,1420,3602,0.596058,0.394225,2.211458
Cleanliness,6,1590,13653,15249,0.104269,0.895337,2.482390
Accuracy of listing,8,1306,4357,5671,0.230294,0.768295,1.911179
Quietness,29,1292,8146,9467,0.136474,0.860463,2.369195
Comfort,16,863,11754,12633,0.068313,0.930420,2.231750
Space and capacity,104,842,8454,9400,0.089574,0.899362,1.877672
Location,151,770,22292,23213,0.033171,0.960324,1.757143
Communication,16,629,18904,19549,0.032176,0.967006,2.465819


## objects inside problem findings.

In [10]:
problem_objects = (
    findings_df[
        (findings_df["aspect"] != "Other")
        & (findings_df["finding_category"] == "Problem")
    ]
    .groupby(["aspect", "object"])
    .size()
    .reset_index(name="problem_count")
    .sort_values(
        ["aspect", "problem_count"],
        ascending=[True, False]
    )
)

problem_objects.head(30)

,aspect,object,problem_count
78,Accuracy of listing,Property,474
35,Accuracy of listing,Listing,316
50,Accuracy of listing,Listing photos,153
68,Accuracy of listing,Overall property,64
83,Accuracy of listing,Property location,42
44,Accuracy of listing,Listing information,36
94,Accuracy of listing,Unit,24
41,Accuracy of listing,Listing description,21
37,Accuracy of listing,Listing address,12
95,Accuracy of listing,Unspecified,12


## top 10 problem objects for each of those six important aspects

In [11]:
priority_aspects = [
    "Amenities",
    "Property condition",
    "Cleanliness",
    "Accuracy of listing",
    "Quietness",
    "Safety and security"
]

top_problem_objects = (
    problem_objects[
        problem_objects["aspect"].isin(priority_aspects)
    ]
    .groupby("aspect", group_keys=False)
    .head(10)
)

top_problem_objects

,aspect,object,problem_count
78,Accuracy of listing,Property,474
35,Accuracy of listing,Listing,316
50,Accuracy of listing,Listing photos,153
68,Accuracy of listing,Overall property,64
83,Accuracy of listing,Property location,42
44,Accuracy of listing,Listing information,36
94,Accuracy of listing,Unit,24
41,Accuracy of listing,Listing description,21
37,Accuracy of listing,Listing address,12
95,Accuracy of listing,Unspecified,12


## largest clear problem, Amenities → Parking

In [12]:
pd.set_option("display.max_colwidth", 150)

findings_df[
    (findings_df["aspect"] == "Amenities")
    & (findings_df["finding_category"] == "Problem")
    & (findings_df["object"] == "Parking")
][
    ["observation", "severity_score", "evidence_quote"]
].sample(20, random_state=42)

,observation,severity_score,evidence_quote
4072,There was a minor issue with parking.,1.0,Minor issue with parking
17112,"Parking is fairly tight, though they were able to make it work due to little traffic.",2.0,"One thing to be aware of is that the parking is fairly tight, but we were able to make it work, as there is little traffic."
22147,Parking can be difficult because the property is in Retro Row and it may be hard to find a close spot after 6pm.,3.0,"Cons: Parking can be difficult, it is right in retro row. If you don’t get home before 6 it may be hard to find a spot close."
193543,"On-site parking was available but the parking space was a little tight, though preferable to street parking.",1.0,parking was a little tight but it’s better than street parking
123096,Finding parking was difficult in the downtown location,2.0,Finding parking was tough so be cautious of that but it’s downtown.
99647,On-site parking is directly in front of the house door and slightly hinders access.,1.0,L'aparcament és just davant de la porta de la casa i dificulta una mica l'accés.
188553,"Only street parking was available, which made finding parking the only bad thing though it wasn't too bad.",2.0,"Only bad thing was finding parking since only street parking was available, but wasn’t too bad."
119256,"It was quite difficult to find a parking spot on the first day, though there were spaces near the house",2.0,it was quite difficult to get a parking spot during the first day but not an issue since there were spaces near the house.
34932,The listing did not offer free on-property parking and paid parking was expensive (about $30/day).,2.0,This air bnb doesn’t offer free parking on property and it’s expensive if you end up paying for it. $30 a day ? Crazy.
194027,Parking is tight but still manageable to get in.,1.0,Parking is tight but you can still manage to get in.


## 8 examples from each of 5 important problem areas

In [13]:
examples_to_check = [
    ("Amenities", "Kitchen"),
    ("Cleanliness", "Property"),
    ("Property condition", "Property"),
    ("Quietness", "Unit"),
    ("Safety and security", "Property location"),
]

samples = []

for aspect_name, object_name in examples_to_check:
    temp = findings_df[
        (findings_df["aspect"] == aspect_name)
        & (findings_df["finding_category"] == "Problem")
        & (findings_df["object"] == object_name)
    ][
        ["aspect", "object", "observation", "severity_score", "evidence_quote"]
    ].sample(
        n=min(8, len(findings_df[
            (findings_df["aspect"] == aspect_name)
            & (findings_df["finding_category"] == "Problem")
            & (findings_df["object"] == object_name)
        ])),
        random_state=42
    )

    samples.append(temp)

problem_samples = pd.concat(samples)

problem_samples

,aspect,object,observation,severity_score,evidence_quote
29451,Amenities,Kitchen,"The kitchen was rudimentary and lacked utensils, though still workable.",2.0,I would have preferred getting more out of the kitchen (ustensile etc). It was fairly rudimentary but still workable.
115033,Amenities,Kitchen,Guest couldn't really cook because there was only one electric stove and not enough space to cook.,3.0,I couldn't really cook in the kitchen because there was only one electrical stove and there was not nought space to cook
175156,Amenities,Kitchen,The kitchen lacked some expected stocked items.,2.0,some minor things the kitchen could have stocked better
28065,Amenities,Kitchen,"The kitchen was attractive and seating was good, but it lacked essentials for cooking (cutting board, more glasses, storage containers, blender, s...",2.0,"The kitchen was beautiful and the seating was great, but lacking some essentials if you actually want to cook. (Cutting board, more glasses, stora..."
134936,Amenities,Kitchen,The kitchen is not very functional,2.0,L’espace cuisine est un peu petit et pas très fonctionnel.
3271,Amenities,Kitchen,"The apartment lacked baking paper and a baking tray, so the guest could not use the oven.",3.0,wenn in der Wohnung noch Backpapier und ein Backblech gewesen wäre. So konnten wir den Ofen in der Küche leider nicht nutzen.
31007,Amenities,Kitchen,The kitchen was poorly equipped and lacked necessary items.,3.0,The kitchen is poorly equipped.
126436,Amenities,Kitchen,Kitchen lacked fresh sealed sponges and needed basic cleaning supplies replaced.,1.0,"Kitchen needs fresh, new, Sealed sponges."
122195,Cleanliness,Property,"There was a spider in the unit, indicating poor cleanliness.",2.0,there was a spider
34929,Cleanliness,Property,The property had a noticeable food smell.,1.0,The place smelled like food.


# Second actual business-analysis question
guest experience changes between 2023, 2024, and 2025

In [14]:
year_summary = (
    findings_df[
        findings_df["aspect"] != "Other"
    ]
    .groupby(["year", "finding_category"])
    .size()
    .unstack(fill_value=0)
)

year_summary["total"] = year_summary.sum(axis=1)

year_summary["problem_rate"] = (
    year_summary["Problem"] / year_summary["total"]
)

year_summary["strength_rate"] = (
    year_summary["Strength"] / year_summary["total"]
)

year_summary

finding_category,Neutral,Problem,Strength,total,problem_rate,strength_rate
year,,,,,,
2023,185,3859,50616,54660,0.070600,0.926015
2024,247,4958,60546,65751,0.075406,0.920838
2025,306,6212,64967,71485,0.086899,0.908820


In [15]:
problem_comment_ids = set(
    findings_df.loc[
        findings_df["finding_category"] == "Problem",
        "comment_id"
    ]
)

reviews_df["has_problem"] = reviews_df["comment_id"].isin(problem_comment_ids)

review_year_summary = (
    reviews_df
    .groupby("year")
    .agg(
        review_count=("comment_id", "size"),
        reviews_with_problem=("has_problem", "sum"),
        average_findings_per_review=("finding_count", "mean")
    )
)

review_year_summary["problem_review_rate"] = (
    review_year_summary["reviews_with_problem"]
    / review_year_summary["review_count"]
)

review_year_summary

,review_count,reviews_with_problem,average_findings_per_review,problem_review_rate
year,,,,
2023,14061,2134,4.105042,0.151767
2024,17142,2560,4.026776,0.149341
2025,18797,2960,3.983348,0.157472


## which guest-experience dimensions actually changed between 2023 and 2025

In [16]:
aspect_year_problem = (
    findings_df[
        (findings_df["aspect"] != "Other")
    ]
    .groupby(["year", "aspect", "finding_category"])
    .size()
    .unstack(fill_value=0)
)

aspect_year_problem["total"] = aspect_year_problem.sum(axis=1)

aspect_year_problem["problem_rate"] = (
    aspect_year_problem["Problem"]
    / aspect_year_problem["total"]
)

aspect_year_problem = aspect_year_problem.reset_index()

aspect_year_problem[
    ["year", "aspect", "Problem", "total", "problem_rate"]
]

finding_category,year,aspect,Problem,total,problem_rate
0,2023,Accuracy of listing,336,1557,0.215800
1,2023,Aesthetics and design,47,3824,0.012291
2,2023,Amenities,820,6932,0.118292
3,2023,Check-in,136,1293,0.105182
4,2023,Check-out,19,259,0.073359
5,2023,Cleanliness,351,4189,0.083791
6,2023,Comfort,251,3699,0.067856
7,2023,Communication,124,5550,0.022342
8,2023,Location,250,6632,0.037696
9,2023,Overall stay,81,10743,0.007540


## a compact 2023-vs-2025 comparison.

In [17]:
year_change = (
    aspect_year_problem[
        aspect_year_problem["year"].isin([2023, 2025])
    ]
    .pivot(
        index="aspect",
        columns="year",
        values="problem_rate"
    )
)

year_change["change_2023_to_2025"] = (
    year_change[2025] - year_change[2023]
)

year_change = year_change.sort_values(
    "change_2023_to_2025",
    ascending=False
)

year_change

year,2023,2025,change_2023_to_2025
aspect,,,
Value for money,0.119380,0.170893,0.051513
Safety and security,0.125311,0.167192,0.041881
Cleanliness,0.083791,0.121733,0.037942
Privacy,0.105791,0.139742,0.033952
Property condition,0.576667,0.609740,0.033073
Check-in,0.105182,0.131941,0.026759
Amenities,0.118292,0.142307,0.024015
Check-out,0.073359,0.095349,0.021990
Accuracy of listing,0.215800,0.237174,0.021374


## listing activity

In [18]:
activity_summary = (
    reviews_df
    .groupby("listing_activity")
    .agg(
        review_count=("comment_id", "size"),
        reviews_with_problem=("has_problem", "sum"),
        average_findings_per_review=("finding_count", "mean")
    )
)

activity_summary["problem_review_rate"] = (
    activity_summary["reviews_with_problem"]
    / activity_summary["review_count"]
)

activity_order = ["Low", "Medium", "High", "Very high"]

activity_summary = activity_summary.reindex(activity_order)

activity_summary

,review_count,reviews_with_problem,average_findings_per_review,problem_review_rate
listing_activity,,,,
Low,733,164,4.321965,0.223738
Medium,2629,437,4.258273,0.166223
High,8934,1477,4.071748,0.165323
Very high,37704,5576,4.001777,0.147889


## which aspects appear as problems most often within each activity group

In [ ]:
activity_aspect_problem = (
    findings_df[
        (findings_df["aspect"] != "Other")
        & (findings_df["finding_category"] == "Problem")
    ]
    .groupby(["listing_activity", "aspect"])["comment_id"]
    .nunique()
    .reset_index(name="reviews_with_problem")
)

activity_aspect_problem["review_count"] = (
    activity_aspect_problem["listing_activity"]
    .map(activity_summary["review_count"])
)

activity_aspect_problem["problem_review_rate"] = (
    activity_aspect_problem["reviews_with_problem"]
    / activity_aspect_problem["review_count"]
)

activity_aspect_problem


In [21]:
activity_aspect_compare = (
    activity_aspect_problem
    .pivot(
        index="aspect",
        columns="listing_activity",
        values="problem_review_rate"
    )
    .reindex(columns=["Low", "Medium", "High", "Very high"])
)

activity_aspect_compare["Low_vs_Very_high"] = (
    activity_aspect_compare["Low"]
    - activity_aspect_compare["Very high"]
)

activity_aspect_compare = activity_aspect_compare.sort_values(
    "Low_vs_Very_high",
    ascending=False
)

activity_aspect_compare

listing_activity,Low,Medium,High,Very high,Low_vs_Very_high
aspect,,,,,
Property condition,0.075034,0.044884,0.041527,0.031853,0.043181
Communication,0.047749,0.020160,0.014551,0.009177,0.038572
Overall stay,0.040928,0.017117,0.010186,0.006657,0.034271
Cleanliness,0.055935,0.035755,0.032124,0.021987,0.033947
Amenities,0.080491,0.060099,0.057981,0.047396,0.033096
Accuracy of listing,0.054570,0.033092,0.027983,0.022968,0.031602
Value for money,0.030014,0.012933,0.008955,0.005702,0.024311
Safety and security,0.028649,0.015976,0.012201,0.010609,0.018040
Quietness,0.040928,0.028528,0.026304,0.023658,0.017270


In [22]:
activity_aspect_compare = activity_aspect_compare.fillna(0)

activity_aspect_compare["Low_vs_Very_high"] = (
    activity_aspect_compare["Low"]
    - activity_aspect_compare["Very high"]
)

activity_aspect_compare = activity_aspect_compare.sort_values(
    "Low_vs_Very_high",
    ascending=False
)

activity_aspect_compare

listing_activity,Low,Medium,High,Very high,Low_vs_Very_high
aspect,,,,,
Property condition,0.075034,0.044884,0.041527,0.031853,0.043181
Communication,0.047749,0.020160,0.014551,0.009177,0.038572
Overall stay,0.040928,0.017117,0.010186,0.006657,0.034271
Cleanliness,0.055935,0.035755,0.032124,0.021987,0.033947
Amenities,0.080491,0.060099,0.057981,0.047396,0.033096
Accuracy of listing,0.054570,0.033092,0.027983,0.022968,0.031602
Value for money,0.030014,0.012933,0.008955,0.005702,0.024311
Safety and security,0.028649,0.015976,0.012201,0.010609,0.018040
Quietness,0.040928,0.028528,0.026304,0.023658,0.017270


# the main strengths for the business story

In [23]:
strength_summary = (
    findings_df[
        (findings_df["aspect"] != "Other")
        & (findings_df["finding_category"] == "Strength")
    ]
    .groupby("aspect")
    .agg(
        strength_count=("aspect", "size"),
        average_strength_score=("aspect_score", "mean")
    )
    .sort_values("strength_count", ascending=False)
)

strength_summary

,strength_count,average_strength_score
aspect,,
Overall stay,37495,4.079904
Location,22292,3.644312
Amenities,20439,3.540486
Communication,18904,4.101619
Cleanliness,13653,3.810078
Aesthetics and design,12636,3.733856
Comfort,11754,3.654926
Space and capacity,8454,3.588242
Quietness,8146,3.544193


In [24]:
review_aspect = (
    findings_df[findings_df["aspect"] != "Other"]
    [["comment_id", "aspect", "finding_category"]]
    .drop_duplicates()
)

aspect_review_summary = (
    review_aspect
    .groupby("aspect")
    .agg(
        reviews_mentioning=("comment_id", "nunique"),
        reviews_with_strength=(
            "finding_category",
            lambda x: (x == "Strength").sum()
        ),
        reviews_with_problem=(
            "finding_category",
            lambda x: (x == "Problem").sum()
        )
    )
)

aspect_review_summary["mention_rate"] = (
    aspect_review_summary["reviews_mentioning"] / 50000
)

aspect_review_summary.sort_values(
    "reviews_mentioning",
    ascending=False
)

,reviews_mentioning,reviews_with_strength,reviews_with_problem,mention_rate
aspect,,,,
Overall stay,37484,36936,417,0.74968
Location,21570,21026,757,0.43140
Communication,18819,18362,564,0.37638
Amenities,16849,15350,2522,0.33698
Cleanliness,14531,13465,1251,0.29062
Aesthetics and design,12182,12050,149,0.24364
Comfort,12117,11440,808,0.24234
Quietness,9228,8099,1232,0.18456
Space and capacity,8864,8125,807,0.17728


## one final master summary

In [25]:
final_aspect_summary = aspect_review_summary.copy()

final_aspect_summary["problem_review_rate"] = (
    final_aspect_summary["reviews_with_problem"] / 50000
)

final_aspect_summary["strength_review_rate"] = (
    final_aspect_summary["reviews_with_strength"] / 50000
)

final_aspect_summary["average_problem_severity"] = (
    problem_severity
)

final_aspect_summary["average_score"] = (
    aspect_summary["average_score"]
)

final_aspect_summary = final_aspect_summary.sort_values(
    "reviews_with_problem",
    ascending=False
)

final_aspect_summary

,reviews_mentioning,reviews_with_strength,reviews_with_problem,mention_rate,problem_review_rate,strength_review_rate,average_problem_severity,average_score
aspect,,,,,,,,
Amenities,16849,15350,2522,0.33698,0.05044,0.30700,1.952824,2.802311
Property condition,3108,1404,1745,0.06216,0.03490,0.02808,2.211458,-0.016380
Cleanliness,14531,13465,1251,0.29062,0.02502,0.26930,2.482390,3.106171
Accuracy of listing,5542,4353,1243,0.11084,0.02486,0.08706,1.911179,2.160113
Quietness,9228,8099,1232,0.18456,0.02464,0.16198,2.369195,2.706137
Comfort,12117,11440,808,0.24234,0.01616,0.22880,2.231750,3.238423
Space and capacity,8864,8125,807,0.17728,0.01614,0.16250,1.877672,3.051277
Location,21570,21026,757,0.43140,0.01514,0.42052,1.757143,3.438763
Safety and security,4077,3549,572,0.08154,0.01144,0.07098,2.982055,2.714962


In [26]:
final_aspect_summary["problem_share_when_mentioned"] = (
    final_aspect_summary["reviews_with_problem"]
    / final_aspect_summary["reviews_mentioning"]
)

final_aspect_summary["strength_share_when_mentioned"] = (
    final_aspect_summary["reviews_with_strength"]
    / final_aspect_summary["reviews_mentioning"]
)

final_aspect_summary[
    [
        "reviews_mentioning",
        "mention_rate",
        "reviews_with_problem",
        "problem_review_rate",
        "problem_share_when_mentioned",
        "average_problem_severity",
        "reviews_with_strength",
        "strength_review_rate",
        "strength_share_when_mentioned",
        "average_score"
    ]
]

,reviews_mentioning,mention_rate,reviews_with_problem,problem_review_rate,problem_share_when_mentioned,average_problem_severity,reviews_with_strength,strength_review_rate,strength_share_when_mentioned,average_score
aspect,,,,,,,,,,
Amenities,16849,0.33698,2522,0.05044,0.149682,1.952824,15350,0.30700,0.911033,2.802311
Property condition,3108,0.06216,1745,0.03490,0.561454,2.211458,1404,0.02808,0.451737,-0.016380
Cleanliness,14531,0.29062,1251,0.02502,0.086092,2.482390,13465,0.26930,0.926640,3.106171
Accuracy of listing,5542,0.11084,1243,0.02486,0.224287,1.911179,4353,0.08706,0.785457,2.160113
Quietness,9228,0.18456,1232,0.02464,0.133507,2.369195,8099,0.16198,0.877655,2.706137
Comfort,12117,0.24234,808,0.01616,0.066683,2.231750,11440,0.22880,0.944128,3.238423
Space and capacity,8864,0.17728,807,0.01614,0.091042,1.877672,8125,0.16250,0.916629,3.051277
Location,21570,0.43140,757,0.01514,0.035095,1.757143,21026,0.42052,0.974780,3.438763
Safety and security,4077,0.08154,572,0.01144,0.140299,2.982055,3549,0.07098,0.870493,2.714962


## save our analysis tables

In [27]:
ANALYSIS_DIR = Path("outputs/business_analysis")
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

final_aspect_summary.reset_index().to_csv(
    ANALYSIS_DIR / "aspect_summary.csv",
    index=False
)

year_change.reset_index().to_csv(
    ANALYSIS_DIR / "year_change.csv",
    index=False
)

activity_aspect_compare.reset_index().to_csv(
    ANALYSIS_DIR / "activity_aspect_compare.csv",
    index=False
)

priority_summary.reset_index().to_csv(
    ANALYSIS_DIR / "problem_priority_summary.csv",
    index=False
)

print("Analysis tables saved to:", ANALYSIS_DIR)

Analysis tables saved to: outputs\business_analysis


## one more useful review-level table

In [28]:
reviews_analysis = reviews_df.copy()

reviews_analysis.to_csv(
    ANALYSIS_DIR / "reviews_50000_analysis.csv",
    index=False
)

print("Saved:", ANALYSIS_DIR / "reviews_50000_analysis.csv")
print("Rows:", len(reviews_analysis))
print("Columns:", reviews_analysis.columns.tolist())

Saved: outputs\business_analysis\reviews_50000_analysis.csv
Rows: 50000
Columns: ['source_row', 'part_number', 'listing_id', 'comment_id', 'date', 'year', 'listing_comment_count', 'listing_activity', 'comments_original', 'comments_clean', 'finding_count', 'batch_id', 'attempt', 'has_problem']
